In [ ]:
from pathlib import Path
from pptx import Presentation
from docx import Document
from unstructured.partition.auto import partition
from langchain.text_splitter import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import requests
import json
import hashlib
import os

c:\Users\zyzai\miniconda3\envs\diss_proj\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def extract_text_from_pptx(file_path):
    try:
        prs = Presentation(file_path)
        text = [shape.text for slide in prs.slides for shape in slide.shapes if hasattr(shape, "text")]
        return "\n".join(text)
    except Exception:
        return None

def extract_text_from_docx(file_path):
    try:
        doc = Document(file_path)
        return "\n".join([para.text for para in doc.paragraphs])
    except Exception:
        return None

def extract_text_with_unstructured(file_path):
    try:
        elements = partition(file_path=file_path)
        return "\n".join([str(el) for el in elements])
    except Exception:
        return None

def extract_text(file_path):
    ext = Path(file_path).suffix.lower()
    if ext == ".pptx":
        text = extract_text_from_pptx(file_path)
    elif ext == ".docx":
        text = extract_text_from_docx(file_path)
    elif ext == ".pdf":
        text = extract_text_with_unstructured(file_path)
    else:
        text = None
    return text or extract_text_with_unstructured(file_path) or ""


In [3]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=150,
    separators=["\n\n", "\n• ", "\n- ", "\n", ". ", " ", ""]
)

base_path = Path("Diss_doc")  # update if needed
all_chunks = []

folders = sorted([p for p in base_path.iterdir() if p.is_dir()], key=lambda p: p.name)
for folder in folders:
    files = sorted([f for f in folder.rglob("*") if f.is_file()], key=lambda f: str(f))
    for file in files:
        if file.suffix.lower() in [".pptx", ".docx", ".pdf"]:
            raw_text = extract_text(file)
            if not raw_text:
                continue
            chunks = text_splitter.split_text(raw_text)

            doc_type = file.suffix.lower().lstrip(".")
            source_rel = f"{folder.name}/{file.name}"
            stat = file.stat()
            mtime = int(stat.st_mtime)
            doc_hash = hashlib.md5((str(file.resolve()) + str(mtime)).encode("utf-8")).hexdigest()

            for idx, chunk in enumerate(chunks):
                if not chunk.strip():
                    continue
                all_chunks.append({
                    "content": chunk,
                    "source": source_rel,
                    "doc_id": doc_hash,
                    "doc_type": doc_type,
                    "folder": folder.name,
                    "filename": file.name,
                    "chunk_idx": idx,
                    "chunk_char_len": len(chunk),
                    "file_mtime": mtime,
                })

In [4]:
model = SentenceTransformer("all-MiniLM-L6-v2")
texts = [c["content"] for c in all_chunks]
metas = [c for c in all_chunks]

embeddings = model.encode(texts, show_progress_bar=True, normalize_embeddings=True)
embeddings = np.asarray(embeddings, dtype="float32")

dim = embeddings.shape[1]
index = faiss.IndexIDMap2(faiss.IndexFlatIP(dim))
ids = np.arange(len(embeddings), dtype=np.int64)
index.add_with_ids(embeddings, ids)

def search(query, top_k=5):
    query_vec = model.encode([query], normalize_embeddings=True)
    query_vec = np.asarray(query_vec, dtype="float32")
    scores, indices = index.search(query_vec, top_k)
    for i, idx in enumerate(indices[0]):
        if idx == -1:
            continue
        m = metas[idx]
        # print(f"\n--- Match {i+1} ---")
        # print(f"Cosine Similarity: {scores[0][i]:.4f}")
        # print(f"Source: {m['source']} | chunk #{m['chunk_idx']} | type={m['doc_type']}")
        # print(f"Content:\n{texts[idx][:500]}")

Batches: 100%|██████████| 33/33 [00:04<00:00,  6.96it/s]


In [5]:
persist_dir = Path("vector_store")
persist_dir.mkdir(parents=True, exist_ok=True)
faiss.write_index(index, str(persist_dir / "docs.index"))
with open(persist_dir / "metas.json", "w", encoding="utf-8") as f:
    json.dump(metas, f, ensure_ascii=False)
np.save(persist_dir / "texts.npy", np.array(texts, dtype=object))

In [6]:
def query_mistral(prompt, model="mistral:latest"):
    r = requests.post(
        "http://localhost:11434/api/generate",
        json={"model": model, "prompt": prompt, "stream": False}
    )
    return r.json()["response"]


In [7]:
def build_prompt(
    query,
    context_chunks_with_sources,
    conversation="",           # kept for compatibility
    mode="initial",            # "initial" | "followup"
    history_snippets=None,
    max_items=8
):
    # Use history_snippets if provided
    if history_snippets:
        # keep last few turns to control size
        conversation = "\n".join(history_snippets[-8:])

    # Build context
    context_blocks = []
    for i, (chunk, src) in enumerate(context_chunks_with_sources, 1):
        context_blocks.append(f"[{src}]\n{chunk}")
    context_text = "\n".join(context_blocks) if context_blocks else "N/A"

    if mode == "initial":
        prompt = f"""You are a senior consultant leading the discovery phase of a client project.

You are given excerpts from past project documents. These are strictly your ONLY source of information. Do NOT use any external knowledge.

Each excerpt starts with a filename in [brackets], followed by content. Stay within the facts only. Do NOT guess, assume, generalize, or invent anything that is not explicitly written.

--- START OF CONTEXT ---
{context_text}
--- END OF CONTEXT ---

Previous conversation:
{conversation if conversation else "N/A"}

Your task:
-make sure to do the following strictly
- List all clear, stated client requirements from the request (not inferred ones).
- Find similar past projects based only on visible matches in the excerpts.
- For each similar project, include:
  - The [filename]
  - A short fact-based summary of what was done
- Mention tools, industries, or outcomes only if they appear directly in the context.
- Do NOT fabricate greetings, summaries, advice, or project plans.
- Output strictly in this bullet list format:
  - [filename]: short factual statement
  - identify the technologies used in similar previous projects
  - industries involved
  - a basic timeframe
"""
    else:  # follow-up mode
        prompt = f"""You are continuing an ongoing consulting discussion.

--- START OF CONTEXT ---
{context_text}
--- END OF CONTEXT ---

Previous conversation:
{conversation if conversation else "N/A"}

Follow-up request:
\"\"\"{query}\"\"\"

Instructions:
- Resolve references like "you said" using the previous conversation first.
- Only consult <doc> excerpts if you need a specific fact.
- Keep it brief and actionable, no restating entire plans.
- Use bullet points, cite [filename] inline only when pulling from context.
- If nothing in conversation or context supports the request, say "N/A".
"""
    return prompt


In [8]:
def _get_source_label(meta):
    if isinstance(meta, dict):
        return meta.get("source", meta.get("filename", "unknown"))
    return str(meta)

def rag(query, top_k=10, min_score=0.5, mode="initial", history_snippets=None):
    qv = model.encode([query], normalize_embeddings=True).astype(np.float32)
    scores, idxs = index.search(qv, top_k)

    pairs, ranked = [], []
    for rank, idx in enumerate(idxs[0]):
        if idx == -1:
            continue
        score = float(scores[0][rank])
        if score < min_score:
            continue
        chunk = texts[idx]
        src_label = _get_source_label(metas[idx])
        pairs.append((chunk, src_label))
        ranked.append((score, src_label))

    # If no doc matches, still allow answering using prior conversation
    has_history = bool(history_snippets and len(history_snippets) > 0)
    if not pairs and not has_history:
        # print("\n Matches Above Threshold:")
        # print("Not enough info")
        return None

    if pairs:
        print("\n Matches Above Threshold:")
        for i, (s, src) in enumerate(ranked, 1):
            print(f"{i:>2}. {src}  |  score={s:.4f}")

    prompt = build_prompt(
        query,
        pairs,
        history_snippets=history_snippets,
        mode=mode,
    )
    resp = query_mistral(prompt)

    print("\n LLM Response:")
    print(resp)
    return resp, pairs, ranked


In [9]:
def _summarize_for_history(text, max_len=220):
    t = " ".join(text.split())
    return t[:max_len] + ("…" if len(t) > max_len else "")


import os, glob

def _delete_faiss(index_prefix_or_file: str):
    for path in glob.glob(index_prefix_or_file + "*"):
        try:
            os.remove(path)
            print("DBs deleted")
        except FileNotFoundError:
            pass

def run_chatbot(top_k_initial=30, top_k_followup=12, min_score=0.5, faiss_index_prefix="./faiss_index"):
    mode = "initial"
    history_snippets = []

    while True:
        user_q = input("\nYou: ").strip()
        if not user_q:
            continue
        if user_q.lower() in {"/quit", "exit", "quit"}:
            if faiss_index_prefix:
                _delete_faiss(faiss_index_prefix)
            print("Bye.")
            break

        tk = top_k_initial if mode == "initial" else top_k_followup
        result = rag(
            user_q,
            top_k=tk,
            min_score=min_score,
            mode=mode,
            history_snippets=history_snippets if mode == "followup" else None,
        )
        if not result:
            continue
        resp, _, _ = result

        history_snippets.append("Client: " + _summarize_for_history(user_q))
        history_snippets.append("Consultant: " + _summarize_for_history(resp))
        mode = "followup"

# Start interactive loop
run_chatbot()



 Matches Above Threshold:
 1. Proj_doc/Copy of Compusoft Business Architecture Discovery Proposal - Clarasys - Final - 04 Nov 2021 - CONFIDENTIAL - v1.1.pptx  |  score=0.5442
 2. Proposals/Copy of Clarasys - Pilgrims_ Friend Society_ Discovery proposal.pptx  |  score=0.5256
 3. Proj_doc/BA&S Previous Project Review and Toolkit - Draft - 2023.pptx  |  score=0.5141
 4. Proposals/Copy of FINAL ANSWERS TO BE REVIEWED FOR FORMATTING.docx  |  score=0.5109
 5. Proposals/Copy of CX Discovery and Transformation Office.pptx  |  score=0.5094
 6. Proposals/Copy of User Centred Design & User Research Sales Deck - Internal - Draft.pptx  |  score=0.5030
 7. Proposals/Copy of Informa Procurement Technology Cost Optimisation - Savings Discovery v2.pptx  |  score=0.5029
 8. Proj_doc/Copy of Clarasys_ Digital Transformation - Future Capability Assessment - DSB shared.pptx  |  score=0.5018
 9. Proj_doc/Copy of Immediate Media Digital Transformation  - Handover deck_.pptx  |  score=0.5018
10. Proj_doc/Cop